In [3]:
import os
import pandas as pd
import numpy as np
from scipy import stats, optimize
import json
import warnings

warnings.filterwarnings('ignore')

print("Библиотеки загружены")

Библиотеки загружены


In [4]:
base_dir = r'C:\Poly\Диплом'
preproc_dir = os.path.join(base_dir, 'Предобработанные данные')
vis_dir = os.path.join(base_dir, 'Визуализация/Глава 4')
os.makedirs(vis_dir, exist_ok=True)

# Файл для сохранения параметров моделей Пуассона
params_file = os.path.join(vis_dir, 'poisson_parameters.json')

print("Папки настроены")

Папки настроены


In [5]:
matches = pd.read_csv(os.path.join(preproc_dir, 'matches_preprocessed.csv'))
matches = matches[matches['league'] == 'La liga'].copy()
matches['date'] = pd.to_datetime(matches['date'])

print("Данные загружены, форма:", matches.shape)

# Разделение на train/test
train = matches[matches['date'] < '2022-01-01'].copy()
test = matches[matches['date'] >= '2022-01-01'].copy()

print(f"Train: {len(train)} матчей, Test: {len(test)} матчей")

Данные загружены, форма: (3420, 57)
Train: 2842 матчей, Test: 578 матчей


In [6]:
all_teams = pd.concat([train['home_team'], train['away_team']]).unique()
team_id = {team: i for i, team in enumerate(all_teams)}
n_teams = len(all_teams)

print(f"Всего уникальных команд в обучающей выборке: {n_teams}")

Всего уникальных команд в обучающей выборке: 31


In [7]:
if os.path.exists(params_file):
    print(f"Найден файл с параметрами: {params_file}")
    with open(params_file, 'r', encoding='utf-8') as f:
        saved = json.load(f)
    
    alpha_p = np.array(saved['alpha_p'])
    beta_p = np.array(saved['beta_p'])
    home_adv_p = saved['home_adv_p']
    alpha_dc = np.array(saved['alpha_dc'])
    beta_dc = np.array(saved['beta_dc'])
    home_adv_dc = saved['home_adv_dc']
    rho_dc = saved['rho_dc']
    
    print("Параметры успешно загружены из файла (оптимизация пропущена)")
    
else:
    print("Файл с параметрами не найден. Запускается полная оптимизация...")

    # ====================== Независимый Пуассон ======================
    def poisson_loglik(params, home_g, away_g, h_idx, a_idx):
        alpha = params[:n_teams]
        beta = params[n_teams:2*n_teams]
        home_adv = params[2*n_teams]
        lambda_h = np.exp(alpha[h_idx] + beta[a_idx] + home_adv)
        lambda_a = np.exp(alpha[a_idx] + beta[h_idx])
        ll = np.sum(stats.poisson.logpmf(home_g, lambda_h) + stats.poisson.logpmf(away_g, lambda_a))
        return -ll

    h_idx_train = train['home_team'].map(team_id).values
    a_idx_train = train['away_team'].map(team_id).values
    home_g = train['home_score'].values
    away_g = train['away_score'].values

    initial = np.zeros(2 * n_teams + 1)
    bounds = [(-5, 5)] * (2 * n_teams) + [(0, 2)]

    res_poisson = optimize.minimize(poisson_loglik, initial,
                                    args=(home_g, away_g, h_idx_train, a_idx_train),
                                    method='L-BFGS-B', bounds=bounds, tol=1e-8)

    alpha_p = res_poisson.x[:n_teams]
    beta_p = res_poisson.x[n_teams:2*n_teams]
    home_adv_p = res_poisson.x[-1]

    print("Независимый Пуассон фит завершен")

    # ====================== Dixon-Coles ======================
    def dixon_coles_loglik(params, home_g, away_g, h_idx, a_idx):
        alpha = params[:n_teams]
        beta = params[n_teams:2*n_teams]
        home_adv = params[2*n_teams]
        rho = params[2*n_teams + 1]
        lambda_h = np.exp(alpha[h_idx] + beta[a_idx] + home_adv)
        lambda_a = np.exp(alpha[a_idx] + beta[h_idx])
        ll = 0.0
        for i in range(len(home_g)):
            lh, la = lambda_h[i], lambda_a[i]
            h, a = home_g[i], away_g[i]
            tau = 1.0
            if h == 0 and a == 0:
                tau = 1 - lh * la * rho
            elif h == 0 and a == 1:
                tau = 1 + lh * rho
            elif h == 1 and a == 0:
                tau = 1 + la * rho
            elif h == 1 and a == 1:
                tau = 1 - rho
            prob = stats.poisson.pmf(h, lh) * stats.poisson.pmf(a, la) * tau
            ll += np.log(max(prob, 1e-12))
        return -ll

    initial_dc = np.append(res_poisson.x, -0.12)
    bounds_dc = [(-5, 5)] * (2 * n_teams) + [(0, 2), (-1.0, 0.5)]

    res_dc = optimize.minimize(dixon_coles_loglik, initial_dc,
                               args=(home_g, away_g, h_idx_train, a_idx_train),
                               method='L-BFGS-B', bounds=bounds_dc,
                               tol=1e-10, options={'maxiter': 1000})

    alpha_dc = res_dc.x[:n_teams]
    beta_dc = res_dc.x[n_teams:2*n_teams]
    home_adv_dc = res_dc.x[2*n_teams]
    rho_dc = res_dc.x[-1]

    print("Dixon-Coles фит завершен, rho =", round(rho_dc, 4))

    # ====================== Сохранение параметров ======================
    saved_params = {
        'alpha_p': alpha_p.tolist(),
        'beta_p': beta_p.tolist(),
        'home_adv_p': float(home_adv_p),
        'alpha_dc': alpha_dc.tolist(),
        'beta_dc': beta_dc.tolist(),
        'home_adv_dc': float(home_adv_dc),
        'rho_dc': float(rho_dc)
    }

    with open(params_file, 'w', encoding='utf-8') as f:
        json.dump(saved_params, f, ensure_ascii=False, indent=4)

    print(f"Параметры успешно сохранены в файл: {params_file}")

Найден файл с параметрами: C:\Poly\Диплом\Визуализация/Глава 4\poisson_parameters.json
Параметры успешно загружены из файла (оптимизация пропущена)


In [8]:
top_teams = ['Real Madrid', 'Barcelona', 'Atletico Madrid', 'Villarreal']

table41_data = []
for team in top_teams:
    if team in team_id:
        idx = team_id[team]
        table41_data.append({
            'Команда': team,
            'Атака (α)': round(alpha_dc[idx], 3),
            'Защита (β)': round(beta_dc[idx], 3)
        })
    else:
        print(f"Предупреждение: команда '{team}' не найдена")

table41 = pd.DataFrame(table41_data)
table41.to_csv(os.path.join(vis_dir, 'Таблица 4.1. Рассчитанные рейтинги силы команд из La Liga.csv'), index=False)

print("\nТаблица 4.1. Рассчитанные рейтинги силы команд из La Liga")
print(table41)


Таблица 4.1. Рассчитанные рейтинги силы команд из La Liga
           Команда  Атака (α)  Защита (β)
0      Real Madrid      0.689      -0.283
1        Barcelona      0.771      -0.357
2  Atletico Madrid      0.308      -0.626
3       Villarreal      0.193      -0.155


In [9]:
def predict_probs(alpha, beta, home_adv, rho, home_team, away_team, dixon=False):
    if home_team not in team_id or away_team not in team_id:
        return 0.333, 0.333, 0.333
    h_idx = team_id[home_team]
    a_idx = team_id[away_team]
    lh = np.exp(alpha[h_idx] + beta[a_idx] + home_adv)
    la = np.exp(alpha[a_idx] + beta[h_idx])
    max_g = 10
    p_h, p_d, p_a = 0.0, 0.0, 0.0
    for hg in range(max_g + 1):
        for ag in range(max_g + 1):
            prob = stats.poisson.pmf(hg, lh) * stats.poisson.pmf(ag, la)
            if dixon:
                tau = 1.0
                if hg == 0 and ag == 0:
                    tau = 1 - lh * la * rho
                elif hg == 0 and ag == 1:
                    tau = 1 + lh * rho
                elif hg == 1 and ag == 0:
                    tau = 1 + la * rho
                elif hg == 1 and ag == 1:
                    tau = 1 - rho
                prob *= tau
            if hg > ag:
                p_h += prob
            elif hg == ag:
                p_d += prob
            else:
                p_a += prob
    total = p_h + p_d + p_a
    if total > 0:
        return p_h/total, p_d/total, p_a/total
    else:
        return 0.333, 0.333, 0.333

In [10]:
def get_metrics(test_df, alpha, beta, home_adv, rho, dixon=False, name=""):
    probs = []
    actual = []
    for _, row in test_df.iterrows():
        ph, pd_, pa = predict_probs(alpha, beta, home_adv, rho, row['home_team'], row['away_team'], dixon)
        probs.append([ph, pd_, pa])
        if row['home_score'] > row['away_score']:
            actual.append(0)
        elif row['home_score'] == row['away_score']:
            actual.append(1)
        else:
            actual.append(2)
    probs = np.array(probs)
    pred = np.argmax(probs, axis=1)
    accuracy = (pred == actual).mean()
    
    cum_pred = np.cumsum(probs, axis=1)
    cum_act = np.zeros((len(actual), 3))
    for i, a in enumerate(actual):
        cum_act[i, a:] = 1
    rps = np.mean(np.sum((cum_pred - cum_act)**2, axis=1) / 2)
    
    logloss = -np.mean(np.log(np.maximum(probs[np.arange(len(actual)), actual], 1e-12)))
    
    return {"Модель": name, 
            "Accuracy (1X2)": round(accuracy, 4),
            "RPS": round(rps, 4), 
            "Log-Loss": round(logloss, 4)}

metrics_poisson = get_metrics(test, alpha_p, beta_p, home_adv_p, 0, False, "Независимый Пуассон")
metrics_dc = get_metrics(test, alpha_dc, beta_dc, home_adv_dc, rho_dc, True, "Диксон-Колз")
metrics_random = {"Модель": "Случайный прогноз", "Accuracy (1X2)": 0.3333, "RPS": 0.2500, "Log-Loss": 1.0986}

table42 = pd.DataFrame([metrics_poisson, metrics_dc, metrics_random])
table42.to_csv(os.path.join(vis_dir, 'Таблица 4.2. Сравнение точности статистических моделей.csv'), index=False)

print("\nТаблица 4.2. Сравнение точности статистических моделей")
print(table42)


Таблица 4.2. Сравнение точности статистических моделей
                Модель  Accuracy (1X2)     RPS  Log-Loss
0  Независимый Пуассон          0.5346  0.2032    0.9913
1          Диксон-Колз          0.5363  0.2031    0.9913
2    Случайный прогноз          0.3333  0.2500    1.0986


In [11]:
league_name = matches['league'].iloc[0]

season_202223 = matches[
    (matches['date'] >= '2022-08-01') &
    (matches['date'] <= '2023-05-31')
].copy()

cutoff = pd.Timestamp('2023-01-15')
played = season_202223[season_202223['date'] < cutoff]
remaining = season_202223[season_202223['date'] >= cutoff]

def calc_points(df):
    if len(df) == 0:
        return pd.Series(dtype=float)
    h_pts = df.groupby('home_team').apply(lambda x: 3 * ((x['home_score'] > x['away_score']).sum()) +
                                          ((x['home_score'] == x['away_score']).sum()))
    a_pts = df.groupby('away_team').apply(lambda x: 3 * ((x['away_score'] > x['home_score']).sum()) +
                                          ((x['away_score'] == x['home_score']).sum()))
    return (h_pts + a_pts.reindex(h_pts.index, fill_value=0)).sort_values(ascending=False)

current_pts = calc_points(played)

n_sim = 10000

champ_count = {t: 0 for t in current_pts.index}
top4_count = {t: 0 for t in current_pts.index}
total_pts_sim = {t: [] for t in current_pts.index}

print(f"\nЗапуск симуляций Монте-Карло для {league_name} (n_sim={n_sim})...")

for sim in range(n_sim):
    sim_pts = current_pts.copy()
    for _, row in remaining.iterrows():
        ph, pd_, pa = predict_probs(alpha_dc, beta_dc, home_adv_dc, rho_dc,
                                    row['home_team'], row['away_team'], True)
        total = ph + pd_ + pa
        if total > 0:
            ph, pd_, pa = ph/total, pd_/total, pa/total
        else:
            ph = pd_ = pa = 1/3.0
            
        outcome = np.random.choice([0, 1, 2], p=[ph, pd_, pa])
        if outcome == 0:
            sim_pts[row['home_team']] += 3
        elif outcome == 1:
            sim_pts[row['home_team']] += 1
            sim_pts[row['away_team']] += 1
        else:
            sim_pts[row['away_team']] += 3

    ranked = sim_pts.sort_values(ascending=False)
    for team in ranked.index[:4]:
        top4_count[team] += 1
    champ_count[ranked.index[0]] += 1
    for team in sim_pts.index:
        total_pts_sim[team].append(sim_pts[team])

table43 = pd.DataFrame({
    'Команда': list(champ_count.keys()),
    'Вер. чемпионства': [round(champ_count[t]/n_sim*100, 1) for t in champ_count],
    'Вер. Топ-4': [round(top4_count[t]/n_sim*100, 1) for t in top4_count],
    'Ср. ожидаемые очки': [round(np.mean(total_pts_sim[t]), 1) for t in total_pts_sim]
})
table43 = table43.sort_values('Вер. чемпионства', ascending=False).head(10)

table43.to_csv(os.path.join(vis_dir, 'Таблица 4.3. Прогноз итоговых позиций (Пример: La Liga 2022-23).csv'), index=False)
print("\nТаблица 4.3. Прогноз итоговых позиций (Пример: La Liga 2022-23)")
print(table43)


Запуск симуляций Монте-Карло для La liga (n_sim=10000)...

Таблица 4.3. Прогноз итоговых позиций (Пример: La Liga 2022-23)
           Команда  Вер. чемпионства  Вер. Топ-4  Ср. ожидаемые очки
0        Barcelona              82.9       100.0                90.2
1      Real Madrid              17.1       100.0                83.8
6  Atletico Madrid               0.1        91.4                70.5
2    Real Sociedad               0.0        58.8                63.8
3       Villarreal               0.0        34.8                61.4
4       Real Betis               0.0         5.7                54.5
5          Osasuna               0.0         0.3                47.1
7    Athletic Club               0.0         5.9                54.9
8   Rayo Vallecano               0.0         0.6                48.1
9         Mallorca               0.0         0.0                42.3
